In [ ]:
import pandas as pd
import requests
import time

# --- Load Input Data ---
gene_name_df = pd.read_csv('/content/drive/MyDrive/Drug Repurposing Project/PlasmoDB_GeneNames_Query_Targets.csv')
target_name_df = pd.read_csv('/content/drive/MyDrive/Drug Repurposing Project/RCSB_Target_Malaria_PDBs.csv')

# --- Shared Utilities ---
def fetch_json(url):
    response = requests.get(url)
    if response.status_code != 200:
        raise Exception(f"Failed to fetch {url} — {response.status_code}")
    return response.json()

def get_structure_summary_df(pdb_id):
    entry_url = f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}"
    entry_data = fetch_json(entry_url)

    polymer_ids = entry_data.get("rcsb_entry_container_identifiers", {}).get("polymer_entity_ids", [])
    ligand_ids = entry_data.get("rcsb_entry_container_identifiers", {}).get("non_polymer_entity_ids", [])

    molecule_list, chains_list, gene_names_list = [], [], []
    organisms_list, lengths_list, mutations_list = [], [], []

    for entity_id in polymer_ids:
        poly = fetch_json(f"https://data.rcsb.org/rest/v1/core/polymer_entity/{pdb_id}/{entity_id}")
        molecule = poly.get("entity", {}).get("pdbx_description", "N/A")
        chains = poly.get("rcsb_polymer_entity", {}).get("pdbx_strand_id", [])
        orgs = [o.get("ncbi_scientific_name", "N/A") for o in poly.get("rcsb_entity_source_organism", [])]
        gene_names = [
            g.get("value") for g in poly.get("rcsb_entity_source_organism", [{}])[0].get("rcsb_gene_name", [])
        ] if poly.get("rcsb_entity_source_organism") else []

        molecule_list.append(molecule)
        chains_list.append(", ".join(chains))
        gene_names_list.append(", ".join(gene_names) if gene_names else "N/A")
        organisms_list.append(", ".join(orgs))
        lengths_list.append(poly.get("entity_poly", {}).get("rcsb_sample_sequence_length", "N/A"))
        mutations_list.append(poly.get("entity_poly", {}).get("rcsb_mutation_count", "N/A"))

    ligand_names = []
    for entity_id in ligand_ids:
        lig = fetch_json(f"https://data.rcsb.org/rest/v1/core/nonpolymer_entity/{pdb_id}/{entity_id}")
        nonpoly = lig.get("pdbx_entity_nonpoly", {})
        name = nonpoly.get("name", "N/A")
        comp_id = nonpoly.get("comp_id", "N/A")
        ligand_names.append(f"{comp_id}: {name}")

    return pd.DataFrame([{
        "PDB_ID": pdb_id,
        "Molecules": molecule_list,
        "Chains": chains_list,
        "Gene_Names": gene_names_list,
        "Organisms": organisms_list,
        "Sequence_Lengths": lengths_list,
        "Mutations": mutations_list,
        "Ligands": ligand_names
    }])

def annotate_structure_info(df, pdb_col_name, prefix):
    lookup_df = []
    for pdb in df[pdb_col_name].dropna():
        try:
            pdb_id = pdb[:4].upper()
            info = get_structure_summary_df(pdb_id)
            lookup_df.append(info)
        except Exception as e:
            print(f"Error fetching info for {pdb}: {e}")

    combined = pd.concat(lookup_df, ignore_index=True)
    lookup = {
        row["PDB_ID"]: {
            "Gene_Names": row["Gene_Names"],
            "Ligands": row["Ligands"]
        }
        for _, row in combined.iterrows()
    }

    df[f"{prefix}_gene_names"] = df[pdb_col_name].str[:4].apply(lambda x: lookup.get(x, {}).get("Gene_Names", "N/A"))
    df[f"{prefix}_ligands"] = df[pdb_col_name].str[:4].apply(lambda x: lookup.get(x, {}).get("Ligands", "N/A"))
    return df


In [ ]:
# --- Clean and Process UniProt / AlphaFold Mapping ---
df1 = gene_name_df.copy()
df1["Accession"] = df1["Accession"].str.extract(r'^(.*?0\d*)')

def get_uniprot_id(raw_query):
    url = "https://rest.uniprot.org/uniprotkb/search"
    params = {"query": raw_query, "format": "json", "fields": "accession", "size": 1}
    response = requests.get(url, params=params)
    if response.status_code != 200:
        return None
    results = response.json().get("results", [])
    return results[0]["primaryAccession"] if results else None

def get_alphafold_filename(uniprot_id):
    return f"AF-{uniprot_id}-F1-model_v4.pdb" if uniprot_id else None

df1["UniProt_ID"] = df1["Accession"].apply(get_uniprot_id)
df1["AlphaFold_File"] = df1["UniProt_ID"].apply(get_alphafold_filename)


In [ ]:
# --- Query RCSB by Gene Name ---
def query_rcsb_by_gene_name(gene_name):
    url = "https://search.rcsb.org/rcsbsearch/v2/query"
    query_payload = {
        "query": {
            "type": "terminal",
            "label": "text",
            "service": "text",
            "parameters": {
                "attribute": "rcsb_entity_source_organism.rcsb_gene_name.value",
                "operator": "exact_match",
                "value": gene_name
            }
        },
        "return_type": "entry",
        "request_options": {
            "paginate": {"start": 0, "rows": 100},
            "results_content_type": ["experimental"],
            "sort": [{"sort_by": "score", "direction": "desc"}],
            "scoring_strategy": "combined"
        }
    }
    response = requests.post(url, json=query_payload)
    if response.status_code != 200:
        print(f"Query failed for {gene_name}")
        return []
    return [entry['identifier'] for entry in response.json().get("result_set", [])]

df1["Gene_PDB"] = df1["Accession"].apply(
    lambda gene: ", ".join(query_rcsb_by_gene_name(gene)) if pd.notna(gene) else "No Match"
)


In [ ]:
# --- Process Target Chains ---
df2 = target_name_df.copy()
df2.drop(columns=['bitscore', 'target_sequence', 'malaria_sequence'], inplace=True)

df2 = annotate_structure_info(df2, 'target_chain_id', prefix='target')

In [ ]:
# --- Process Malaria Chains ---
df2 = annotate_structure_info(df2, 'malaria_match_id', prefix='malaria_match_id')